## Multi-Turn RL with SageMaker AI, part 1: build and deploy the rollout agent

Production LLM agents rarely answer in one shot: they call tools, read results, and adjust course
over many turns. **Multi-turn reinforcement learning (MTRL)** on Amazon SageMaker AI optimizes the
whole trajectory, not just the final answer.

This three-notebook series is a complete, working MTRL example. You fine-tune
`openai-reasoning-gpt-oss-20b` to follow a Standard Operating Procedure (SOP) on the
[SOP-Bench](https://github.com/amazon-science/SOP-Bench) `aircraft_inspection` task, with a
rollout agent hosted on **Amazon Bedrock AgentCore**. Every piece - environment, tools, reward,
agent - is real, runnable code in [`sopbench_agent/`](sopbench_agent/) that you can adapt to your
own task.

| # | Notebook | What you do |
|---|---|---|
| 1 | `01_build_and_deploy_agent.ipynb` | Explore the task, tools, and reward; deploy the rollout agent to Amazon Bedrock AgentCore |
| 2 | `02_train.ipynb` | Build the prompt dataset and launch a GRPO training job with `MultiTurnRLTrainer`; read MLflow |
| 3 | `03_evaluate_and_deploy.ipynb` | Compare base vs. fine-tuned with `MultiTurnRLEvaluator`, diagnose failures, deploy, clean up |

**In this notebook** you will:
1. Deploy the prerequisites and install dependencies.
2. Explore the task, its tools, and a real task instance.
3. Understand how the agent participates in the training loop and connects to the MTRL service.
4. Check that the reward is trustworthy - and see the reward hack it guards against.
5. Deploy the agent to AgentCore.

> **Note:** This is sample code, for non-production usage. Work with your security and legal teams
> to meet your organizational security, regulatory, and compliance requirements before deployment.

![SageMaker AI MTRL service overview: your agent RL environment (on AgentCore, EKS, EC2, Fargate, or custom) talks to the SageMaker AI MTRL service through the Job Runtime APIs (Sample, SampleWithResponseStream, CompleteRollout, UpdateReward). The MTRL service runs training and evaluation, driven by the MultiTurnRLTrainer, MultiTurnRLEvaluator, and ModelBuilder SDKs, and emits a Model Package, checkpoints, CloudWatch logs, and MLflow metrics.](images/service-overview.jpg)

SageMaker AI runs the training loop and, each step, asks your **agent** for a group of rollouts.
You supply three things: the **environment** (the task and its tools), the **reward function**, and
the **agent** that ties them together. The agent calls the policy model served by the training job
(`Sample` / `SampleWithResponseStream`), executes tools, and reports the reward
(`CompleteRollout` / `UpdateReward`) - the `@sagemaker_rft_handler` decorator does the reporting
for you.

### 1. Prerequisites

**Account and region.** Use a sandbox/dev account in a region where SageMaker AI multi-turn RL and
Amazon Bedrock AgentCore are available (this example was tested in `us-east-1`). Set `AWS_REGION`
(or your default region) to that region before you start. The training job role is intentionally
broad *within* its account and region (see the comments in `prerequisites.yaml`).

**Deploy the prerequisites stack** once per account. [`prerequisites.yaml`](prerequisites.yaml)
creates the MTRL training job role, the AgentCore agent execution role, a private KMS-encrypted S3
bucket for prompts and output, a managed MLflow app, and two managed policies:

```bash
aws cloudformation deploy --stack-name mtrl-prerequisites \
  --template-file prerequisites.yaml --capabilities CAPABILITY_NAMED_IAM
# Accounts allow 3 MLflow apps by default. To reuse one instead of creating one, add:
#   --parameter-overrides ExistingMlflowAppArn=<mlflow-app ARN>
```

**Permissions for whoever runs these notebooks** (for example your SageMaker Studio execution
role): attach `AmazonSageMakerFullAccess`, `BedrockAgentCoreFullAccess`, and the two managed
policies the stack outputs as `NotebookPolicyArn` and `AgentCoreToolkitPolicyArn`.

**Python environment.** Python 3.10+ (3.11 recommended). Docker is *not* required:
`agentcore deploy` builds the image on AWS CodeBuild. Install the dependencies, then **restart the
kernel**.

In [ ]:
%pip install -q -r sopbench_agent/requirements-dev.txt
%pip install -q --no-deps -e sopbench_agent

### 2. Configuration

Every account-specific value is read from the prerequisites stack outputs - nothing to paste.

In [ ]:
import os
import sys
from pathlib import Path

import boto3
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer

STACK_NAME = "mtrl-prerequisites"  # the prerequisites.yaml stack you deployed
AGENT_NAME = "sopbench_agentcore"  # the name passed to `agentcore configure --name`
PREFIX = "sopbench-mtrl/aircraft"  # S3 prefix for the prompt parquets
AGENT_DIR = Path("sopbench_agent").resolve()

# Make `!python` / `!agentcore` resolve to this kernel's environment.
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"

sess = boto3.session.Session()
REGION = sess.region_name
assert REGION, "Set AWS_REGION (or a default region) to the region you deployed the stack in."
# The training job must run in the AgentCore runtime's region: pin it for every SDK client.
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"] = REGION
account = sess.client("sts").get_caller_identity()["Account"]

stack = sess.client("cloudformation").describe_stacks(StackName=STACK_NAME)["Stacks"][0]
outs = {o["OutputKey"]: o["OutputValue"] for o in stack.get("Outputs", [])}
BUCKET = outs["PromptsBucketName"]
ROLE_ARN = outs["JobRoleArn"]
AGENTCORE_EXEC_ROLE_ARN = outs["AgentCoreExecRoleArn"]
MLFLOW_APP_ARN = outs["MlflowAppArn"]
TRAIN_DATASET = f"s3://{BUCKET}/{PREFIX}/train/aircraft_inspection_train.parquet"
VAL_DATASET = f"s3://{BUCKET}/{PREFIX}/validation/aircraft_inspection_val.parquet"


def find_agent_runtime_arn(name=AGENT_NAME):
    """ARN of the READY AgentCore runtime with exactly this name (others may share a prefix)."""
    for runtime in MultiTurnRLTrainer.list_bedrock_agentcore_runtimes(session=sess):
        if runtime["name"] == name and runtime["status"] == "READY":
            return runtime["arn"]
    return None


print(f"account={account} region={REGION} bucket={BUCKET}")

### 3. The environment: SOP-Bench `aircraft_inspection`

The task models a pre-flight airworthiness inspection. The agent receives an SOP - a multi-step
protocol for verifying an aircraft's mechanical systems, electrical systems, and component
authenticity - plus input parameters for one aircraft (tail number, component serial numbers,
battery status, weights, ...). It must **follow the SOP step by step, calling tools to gather
information**, then emit a structured report with six decision fields.

Why it is a good RL environment:

* **Multi-turn by nature** - 3-7 tool calls per instance; not solvable in one shot.
* **Many decision fields** - six output fields give natural partial credit and enough variance
  across rollouts for GRPO to learn from.
* **Conditional logic** - incident and mismatch reporting only apply when something fails.
* **Cheap and deterministic** - tools are in-memory lookups, so a rollout takes seconds and is
  reproducible. A good RL environment is cheap, reproducible, and representative of production -
  built from the production tools, not pointed at production.

Load one real instance from the installed `amazon-sop-bench` package:

In [ ]:
import json
from sopbench_agentcore.samples import load_instance

instance = load_instance("aircraft_inspection", task_id=110)
print(instance["instance_id"])
print("task_input  :", json.dumps(json.loads(instance["task_input"]), indent=2))
print("ground_truth:", instance["ground_truth"])

This is what the trainer sends the agent as `payload["prompt"]` (as a JSON string). The agent only
*uses* `task_input`; `ground_truth` is held out and only read by the reward after the rollout.
`nan` marks a field that does not apply to this aircraft - the model must still emit it as `"NA"`.

The seven tools the agent can call, straight from the benchmark:

In [ ]:
from amazon_sop_bench.benchmarks import load_benchmark

benchmark = load_benchmark("aircraft_inspection")
for spec in benchmark.tools.get_tool_specs():
    s = spec["toolSpec"]
    print(f"{s['name']:<28} {s['description'][:90]}")

### 4. How the agent works

The agent is a small package in [`sopbench_agent/src/sopbench_agentcore/`](sopbench_agent/src/sopbench_agentcore/):

| File | Role |
|---|---|
| `main.py` | AgentCore entrypoint; parses the prompt, runs the loop, scores, returns `{"reward": ...}` |
| `openai_agent.py` | The multi-turn tool-use loop over an OpenAI-compatible client |
| `tools.py` | Adapts SOP-Bench tools into function-calling specs and an executor |
| `prompts.py` | System prompt and the output contract |
| `reward.py` | Scores the final answer field by field (dense, task success, and the training reward) |
| `config.py` | Environment-driven settings (turn cap, sampling fallbacks, debug flags) |

The entrypoint stacks two decorators - one per platform:

```python
from bedrock_agentcore import BedrockAgentCoreApp
from sagemaker.train.rft import sagemaker_rft_handler

app = BedrockAgentCoreApp()

@app.entrypoint            # AgentCore: route the HTTP request to this function
@sagemaker_rft_handler     # MTRL: report the returned "reward" back to the trainer
def invoke(payload: dict) -> dict:
    return run_rollout(payload)
```

**The loop.** Seed the conversation with the SOP (system) and the task (user), then repeat: call
the policy model; if it returns tool calls, execute them and append the results; if it returns
plain text, that is the final answer. Stop when the model stops calling tools or hits `max_turns`.
A bad tool call is fed back to the model as an error string so it can self-correct. The shipped
loop also resamples empty responses (and fails the rollout if they persist, so the trainer retries it),
recovers an answer GPT-OSS writes only in its reasoning channel on a turn without tool calls, and treats a `400`/`500` from the
endpoint as the end of the rollout (scoring what was captured).

**Connecting to the MTRL service.** The agent never hosts a model: the training job serves the
policy model and injects its endpoint per rollout in `payload["metadata"]["endpoint"]` (sampling
params in `payload["inferenceParams"]`). The agent authenticates with a short-lived token - no
static keys:

```python
from sagemaker.core.token_generator import generate_token
from sagemaker.train.rft import get_inference_headers

client = OpenAI(base_url=endpoint + "/v1",
                api_key=generate_token(region=region),
                default_headers=get_inference_headers())
```

Each `client.chat.completions.create(...)` is a `Sample` call on the SageMaker AI Job Runtime
(`job-runtime.sagemaker.{region}.api.aws`). The contract is fixed - **accept a prompt, run a
multi-turn rollout, return a reward** - so the trainer only ever needs the agent's runtime ARN.

### 5. The reward - and the reward hack it prevents

`reward.py` scores the rollout's **last tagged answer** against the ground truth, field by field,
with SOP-Bench's own parser and value comparison, so scores line up with the published benchmark:

* **`dense`** - fraction of fields correct (0-1): partial credit gives a usable gradient early on.
* **TSR** (task success rate) - 1 only if *every* field is correct: the metric you report.
* The **training reward** is `dense x 5`, so it ranges from **0** (all wrong) to **5** (perfect);
  a rollout that never emits a tagged answer scores **-1**. Only the last answer counts, so
  repeating an answer earns nothing extra.

**The scar this guards against.** An earlier agent learned to drop the required answer wrapper,
and a lenient reward parser still accepted it: training reward went **up** while the evaluator
scored it **down**. Here an untagged answer is simply no answer (-1), and `prompts.py` requires
**every** expected field (`"NA"` when not applicable).

**Matching rules.** Values compare case-insensitively, null-like values (`"NA"`, `null`) are
equivalent, and - as in SOP-Bench - a value that contains the expected one also matches, so
`"failed (low battery)"` counts as `"failed"`. That containment rule would also let one answer that
lists every option (`"success fail retest"`) collect credit on every task, so the reward adds one
guard: a value that matches **more than one** of the field's allowed values never matches.
Containment is otherwise kept for benchmark parity, which also means a negation such as
`"unsuccessful"` still matches `"success"`.

**Tag names matter for GPT-OSS.** The prompt asks for the answer as
`<task_result><answer_json>{...}</answer_json></task_result>` - not SOP-Bench's native
`<final_output>`. gpt-oss confuses a tag containing the word "final" with its Harmony `final`
channel and writes the answer into the reasoning channel, so the endpoint returns an empty message.
Renaming the tags cut empty responses from 8.7% to 0.2%. `reward.py` maps them back before scoring,
and still accepts SOP-Bench's own formats.

Score four canned answers against the instance above (`scripts/inspect_reward.py`; no model or AWS
calls):

In [ ]:
!cd {AGENT_DIR} && PYTHONPATH=src python scripts/inspect_reward.py

* **Case 2** loses credit and fails TSR: every expected field must be emitted.
* **Case 3** loses that one field: a value that matches several allowed values never counts.
* **Case 4** has the right values but no answer tags, so the rollout has not answered: **-1**.

Run the offline tests of the agent loop, the reward, and the entrypoint (fake model client, no
endpoint or AWS calls):

In [ ]:
!cd {AGENT_DIR} && PYTHONPATH=src python scripts/test_local.py

### 6. Deploy the agent to AgentCore

`agentcore configure` writes a local `.bedrock_agentcore.yaml` pointing at the execution role from
the prerequisites stack (it has `AmazonSageMakerJobRuntimeAccess`, which the agent needs to call
`Sample`, `CompleteRollout`, and `UpdateReward`). `agentcore deploy` then builds the arm64 image on
CodeBuild, pushes it to ECR, and creates the runtime - a few minutes. No local container engine
is needed; if `configure` stalls at a `Password:` prompt, a locally installed engine (such as Finch)
is asking for `sudo` while the CLI probes for it - run the command from a terminal instead.

In [ ]:
!cd {AGENT_DIR} && AGENTCORE_SUPPRESS_RECOMMENDATION=1 agentcore configure --non-interactive \
    --entrypoint src/sopbench_agentcore/main.py --name {AGENT_NAME} \
    --requirements-file requirements.txt --region {REGION} --deployment-type container \
    --protocol HTTP --disable-memory --execution-role {AGENTCORE_EXEC_ROLE_ARN}

In [ ]:
!cd {AGENT_DIR} && AGENT_NAME={AGENT_NAME} ./scripts/deploy.sh

In [ ]:
import time

for _ in range(60):
    AGENT_RUNTIME_ARN = find_agent_runtime_arn()
    if AGENT_RUNTIME_ARN:
        break
    time.sleep(20)
print("AGENT_RUNTIME_ARN =", AGENT_RUNTIME_ARN)
assert AGENT_RUNTIME_ARN, f"No READY runtime named {AGENT_NAME}; check the deploy output above."

Runtime logs land in CloudWatch under `/aws/bedrock-agentcore/runtimes/<runtime-id>-DEFAULT`. To
debug a rollout, redeploy with `DEBUG=1 ./scripts/deploy.sh` for verbose per-turn logs (large log
volume - not for full training runs).

#### AgentCore notes

* **arm64-only runtime.** Don't pin `--platform=linux/arm64` in the Dockerfile; CodeBuild builds
  arm64, and `agentcore deploy --local` builds native amd64 for local testing.
* **2048 MB image cap (non-adjustable).** The image stays under it by using the modular
  `sagemaker-train` + `sagemaker-core` packages and a multi-stage Dockerfile.
* **Region.** The training job must run in the runtime's region.
* **Concurrency.** AgentCore has no server-side cap, so the trainer's `rollout_max_concurrency` is
  the *only* throttle on concurrent inference connections (more in notebook 2).
* **Other hosting.** The same contract works for agents on Amazon EKS, Amazon EC2, AWS Fargate, or
  your own infrastructure, connected through a Lambda forwarder - see
  [Preparing your agent](https://docs.aws.amazon.com/sagemaker/latest/dg/model-customize-mtrl-agent.html).

**Next:** `02_train.ipynb` builds the prompt dataset and launches training against this runtime.